In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

if not os.path.exists('/content/RecoSys'):
    !git clone -b dev https://github.com/sbnikhil/RecoSys.git /content/RecoSys

%cd /content/RecoSys
!pip install pandas pyarrow numpy huggingface_hub -q

Cloning into '/content/RecoSys'...
remote: Enumerating objects: 532, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 532 (delta 76), reused 156 (delta 54), pack-reused 338 (from 1)
Receiving objects: 100% (532/532), 1.48 MiB | 24.38 MiB/s, done.
Resolving deltas: 100% (253/253), done.
/content/RecoSys


In [4]:
from huggingface_hub import hf_hub_download
import shutil, os

v = hf_hub_download(repo_id="manojarulmurugan/recosys-models",
                    filename="vocabs.pkl", repo_type="dataset")
os.makedirs("model", exist_ok=True)
shutil.copy(v, "model/vocabs.pkl")
print("vocabs.pkl ready:", round(os.path.getsize("model/vocabs.pkl")/1e6, 1), "MB")

vocabs.pkl:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

vocabs.pkl ready: 21.8 MB


In [5]:
import shutil, os, time

src = "/content/drive/MyDrive/rees46"
dst = "/content/rees46_local"
os.makedirs(dst, exist_ok=True)

for f in ["2019-Oct.csv.gz","2019-Nov.csv.gz","2019-Dec.csv.gz","2020-Jan.csv.gz","2020-Feb.csv.gz"]:
    d = f"{dst}/{f}"
    if os.path.exists(d):
        print(f"  Already local: {f}"); continue
    s = f"{src}/{f}"
    if not os.path.exists(s):
        s = s.replace(".csv.gz", ".csv"); d = d.replace(".csv.gz", ".csv")
    if not os.path.exists(s):
        print(f"  NOT FOUND: {f}"); continue
    sz = os.path.getsize(s)/1e6
    print(f"  Copying {f} ({sz:.0f} MB)...", end=" ", flush=True)
    t = time.time()
    shutil.copy2(s, d)
    print(f"done in {time.time()-t:.0f}s")

print("Local files:", sorted(os.listdir(dst)))


  Copying 2019-Oct.csv.gz (1742 MB)... done in 49s
  Copying 2019-Nov.csv.gz (2890 MB)... done in 79s
  Copying 2019-Dec.csv.gz (2952 MB)... done in 81s
  Copying 2020-Jan.csv.gz (2390 MB)... done in 74s
  Copying 2020-Feb.csv.gz (2347 MB)... done in 66s
Local files: ['2019-Dec.csv.gz', '2019-Nov.csv.gz', '2019-Oct.csv.gz', '2020-Feb.csv.gz', '2020-Jan.csv.gz']


In [6]:
!python scripts/sequence/rebuild_local_sessions.py \
    --events-dir /content/rees46_local \
    --vocabs-path model/vocabs.pkl \
    --out-dir /content/drive/MyDrive/recosys_sessions

RecoSys — Rebuild session parquets from local REES46 CSVs
  Events dir  : /content/rees46_local
  Vocabs path : model/vocabs.pkl
  Output dir  : /content/drive/MyDrive/recosys_sessions
  VAL start   : 2020-01-25
  TEST start  : 2020-02-01

  ▶ Loading vocabs from model/vocabs.pkl ...
     done in 0m 0s — 890,736 users  /  222,863 items

  ▶ Reading 2019-Oct.csv.gz in chunks of 200,000 ...
     done in 6m 15s — 42,448,764 raw rows → 5,490,970 in-vocab rows kept

  ▶ Reading 2019-Nov.csv.gz in chunks of 200,000 ...
     done in 9m 37s — 67,501,979 raw rows → 8,770,842 in-vocab rows kept

  ▶ Reading 2019-Dec.csv.gz in chunks of 200,000 ...
     done in 9m 24s — 67,542,878 raw rows → 8,703,645 in-vocab rows kept

  ▶ Reading 2020-Jan.csv.gz in chunks of 200,000 ...
     done in 7m 52s — 55,967,041 raw rows → 7,139,619 in-vocab rows kept

  ▶ Concatenating train-window months ...
     done in 0m 1s — 30,105,076 total events

  Event split:
     train (before 2020-01-25): 28,491,966 events


In [7]:
import pandas as pd
for f in ["train_sessions.parquet", "val_sessions.parquet", "test_sessions.parquet"]:
    df = pd.read_parquet(f"/content/drive/MyDrive/recosys_sessions/{f}")
    print(f"{f}: {len(df):,} sessions")


train_sessions.parquet: 2,887,783 sessions
val_sessions.parquet: 151,693 sessions
test_sessions.parquet: 515,358 sessions
